# DX 603: Project Milestone Two: Modeling and Feature Engineering

### Due: Sunday July 26 @ 11:59PM (with grace period of 2 hours & 1 minute)

### Overview

In Milestone 1, you explored the Zillow dataset, cleaned the data, and developed hypotheses about how preprocessing and feature engineering might improve predictive performance.

In this milestone, you will  develop, evaluate, and refine several machine learning models using those ideas. Rather than simply searching for the best algorithm, you will follow an iterative modeling workflow by:

1. Establishing baseline performance using several regression models.
2. Testing the preprocessing and feature engineering ideas proposed in Milestone 1.
3. Refining the feature set through feature selection.
4. Optimizing model performance through hyperparameter tuning.
5. Comparing the evolution of your models and selecting a final model to evaluate on the held-out test set.

Throughout this milestone, use **repeated 5-fold cross-validation (5 repeats)** to guide your modeling decisions. The held-out test set should be used only once, after all modeling decisions have been completed.




In [1]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV,
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, HistGradientBoostingRegressor

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))



In [2]:

url = "https://www.cs.bu.edu/fac/snyder/cs505/Data/zillow_dataset.csv"

filename = os.path.basename(urlparse(url).path)

if not os.path.exists(filename):
    try:
        print("Downloading the file...")
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad status codes
        with open(filename, "wb") as f:
            f.write(response.content)
        print("File downloaded successfully.")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading the file: {e}")
else:
    print("File already exists. Skipping download.")

df = pd.read_csv(filename)

File downloaded successfully.


In [3]:
#Final lost of columns to be dropped based on above three steps:
cols_to_drop= [
    'buildingclasstypeid',
    'finishedsquarefeet13',
    'basementsqft',
    'storytypeid',
    'yardbuildingsqft26',
    #'fireplaceflag',
    'architecturalstyletypeid',
    'typeconstructiontypeid',
    'finishedsquarefeet6',
    'pooltypeid10',
    'decktypeid',
    #'poolsizesum',
    'pooltypeid2',
    'hashottuborspa',
    'taxdelinquencyyear',
    #'taxdelinquencyflag',
    'finishedsquarefeet15',
'parcelid', 'fips', 'assessmentyear', 'regionidcounty',#'rawcensustractandblock', 'censustractandblock',
                    #'hashottuborspa'
                    'pooltypeid7','poolcnt']

df_reduced = df.drop(columns=cols_to_drop)
# Remove Problematic Samples
df_clean = df_reduced.copy()

# 1. Remove data samples with missing target values
df_clean = df_clean[df_clean["taxvaluedollarcnt"].notna()]

# 2. Remove data samples with >90% missing features
row_missing_pct = df_clean.isna().mean(axis=1) * 100
df_clean = df_clean[row_missing_pct <= 90]

print("Remaining rows after cleaning:", df_clean.shape[0])
# Verify the new shape
print("Original shape:", df.shape)
print("New shape:", df_clean.shape)

Remaining rows after cleaning: 77578
Original shape: (77613, 55)
New shape: (77578, 35)


In [4]:
# Split the Dataset into Training and Test Sets
X = df_clean.drop(columns=["taxvaluedollarcnt"])
y = df_clean["taxvaluedollarcnt"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (62062, 34)
Test shape: (15516, 34)


In [5]:
# 1. Identify column types
ordinal_cols = ['buildingqualitytypeid']

categorical_cols = [
    'airconditioningtypeid',
    'heatingorsystemtypeid',
    'propertycountylandusecode',
    'propertylandusetypeid',
    'propertyzoningdesc',
    'regionidcity',
    'regionidneighborhood',
    'regionidzip',
    'fireplaceflag',
    'taxdelinquencyflag']

numeric_cols = [col for col in X_train.columns
                if col not in categorical_cols + ordinal_cols]

# 2. Fit imputers on TRAIN only

# Numeric imputer (median)
num_imputer = SimpleImputer(strategy="median")
num_imputer.fit(X_train[numeric_cols])

# Ordinal imputer (median)
ord_imputer = SimpleImputer(strategy="median")
ord_imputer.fit(X_train[ordinal_cols])

# Categorical imputer ("Unknown")
cat_imputer = SimpleImputer(strategy="constant", fill_value="Unknown")
cat_imputer.fit(X_train[categorical_cols])

# 3. Transform TRAIN + TEST
# Numeric
X_train_num = pd.DataFrame(
    num_imputer.transform(X_train[numeric_cols]),
    columns=numeric_cols,
    index=X_train.index)

X_test_num = pd.DataFrame(
    num_imputer.transform(X_test[numeric_cols]),
    columns=numeric_cols,
    index=X_test.index)

# Ordinal
X_train_ord = pd.DataFrame(
    ord_imputer.transform(X_train[ordinal_cols]),
    columns=ordinal_cols,
    index=X_train.index)

X_test_ord = pd.DataFrame(
    ord_imputer.transform(X_test[ordinal_cols]),
    columns=ordinal_cols,
    index=X_test.index)

# Categorical
X_train_cat = pd.DataFrame(
    cat_imputer.transform(X_train[categorical_cols]),
    columns=categorical_cols,
    index=X_train.index)

X_test_cat = pd.DataFrame(
    cat_imputer.transform(X_test[categorical_cols]),
    columns=categorical_cols,
    index=X_test.index)

# 4. Final combined datasets
X_train_imputed = pd.concat([X_train_num, X_train_ord, X_train_cat], axis=1)
X_test_imputed = pd.concat([X_test_num, X_test_ord, X_test_cat], axis=1)

print("Missing values in TRAIN:", X_train_imputed.isna().sum().sum())
print("Missing values in TEST:", X_test_imputed.isna().sum().sum())


Missing values in TRAIN: 0
Missing values in TEST: 0


In [6]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler

ordinal_cols = ['buildingqualitytypeid']

high_card_cols = [
    'regionidcity',
    'regionidneighborhood',
    'regionidzip',
    'propertycountylandusecode',
    'propertylandusetypeid',
    'propertyzoningdesc'
]

low_card_cols = [
    'airconditioningtypeid',
    'heatingorsystemtypeid'
]

boolean_cols = [
    'fireplaceflag',
    'taxdelinquencyflag'
]

numeric_cols = [
    col for col in X_train_imputed.columns
    if col not in ordinal_cols + high_card_cols + low_card_cols + boolean_cols
]

convert_to_str = high_card_cols + low_card_cols + boolean_cols

X_train_imputed[convert_to_str] = X_train_imputed[convert_to_str].fillna("Unknown").astype(str)
X_test_imputed[convert_to_str]  = X_test_imputed[convert_to_str].fillna("Unknown").astype(str)

# Ordinal stays numeric
X_train_imputed[ordinal_cols] = X_train_imputed[ordinal_cols].fillna(-1)
X_test_imputed[ordinal_cols]  = X_test_imputed[ordinal_cols].fillna(-1)

ord_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
ord_enc.fit(X_train_imputed[ordinal_cols])

X_train_ord = pd.DataFrame(ord_enc.transform(X_train_imputed[ordinal_cols]),
                           columns=ordinal_cols, index=X_train_imputed.index)

X_test_ord = pd.DataFrame(ord_enc.transform(X_test_imputed[ordinal_cols]),
                          columns=ordinal_cols, index=X_test_imputed.index)
def frequency_encode(train, test, cols):
    for col in cols:
        freq = train[col].value_counts() / len(train)
        train[col] = train[col].map(freq)
        test[col]  = test[col].map(freq).fillna(0)
    return train, test

X_train_freq = X_train_imputed[high_card_cols].copy()
X_test_freq  = X_test_imputed[high_card_cols].copy()

X_train_freq, X_test_freq = frequency_encode(X_train_freq, X_test_freq, high_card_cols)

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(X_train_imputed[low_card_cols])

X_train_ohe = pd.DataFrame(ohe.transform(X_train_imputed[low_card_cols]),
                           columns=ohe.get_feature_names_out(low_card_cols),
                           index=X_train_imputed.index)

X_test_ohe = pd.DataFrame(ohe.transform(X_test_imputed[low_card_cols]),
                          columns=ohe.get_feature_names_out(low_card_cols),
                          index=X_test_imputed.index)
bool_map = {
    "Y": 1, "N": 0, "Unknown": -1,
    "True": 1, "False": 0,
    True: 1, False: 0
}

X_train_bool = X_train_imputed[boolean_cols].replace(bool_map)
X_test_bool  = X_test_imputed[boolean_cols].replace(bool_map)

X_train_numeric = X_train_imputed[numeric_cols]
X_test_numeric  = X_test_imputed[numeric_cols]

/tmp/ipykernel_5015/736878464.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train_imputed[convert_to_str] = X_train_imputed[convert_to_str].fillna("Unknown").astype(str)
/tmp/ipykernel_5015/736878464.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_test_imputed[convert_to_str]  = X_test_imputed[convert_to_str].fillna("Unknown").astype(str)
/tmp/ipykernel_5015/736878464.py:74: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_obje

In [7]:
# Replace original flag columns with numeric versions
X_train_imputed[boolean_cols] = X_train_imputed[boolean_cols].replace(bool_map)
X_test_imputed[boolean_cols]  = X_test_imputed[boolean_cols].replace(bool_map)
X_train_final = pd.concat([
    X_train_numeric,
    X_train_ord,
    X_train_freq,
    X_train_ohe,
    X_train_imputed[boolean_cols]   # numeric flags
], axis=1)

X_test_final = pd.concat([
    X_test_numeric,
    X_test_ord,
    X_test_freq,
    X_test_ohe,
    X_test_imputed[boolean_cols]    # numeric flags
], axis=1)

non_numeric = X_train_final.select_dtypes(include=['object'])
print(non_numeric.columns.tolist())

[]


/tmp/ipykernel_5015/716909793.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train_imputed[boolean_cols] = X_train_imputed[boolean_cols].replace(bool_map)
/tmp/ipykernel_5015/716909793.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_test_imputed[boolean_cols]  = X_test_imputed[boolean_cols].replace(bool_map)


## Prelude: Load Your Preprocessed Dataset from Milestone 1

In Milestone 1, you cleaned the Zillow dataset by removing unsuitable features, handling missing values, and encoding categorical variables. In this milestone, you will build, compare, and improve several regression models using that prepared dataset.

Begin by returning to your Milestone 1 notebook and rerunning your code through Part 3, where your dataset has been completely cleaned and encoded, but before any experimental feature engineering ideas were evaluated. Save these datasets to use as the starting point for this milestone.

For example, do this at the end of Milestone 1:

```python
X_train.to_csv("X_train.csv", index=False)               # or whatever names you gave these sets
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)
```

Then load them at the beginning of the Milestone 2 notebook:

```python
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")

y_train = pd.read_csv("y_train.csv").squeeze("columns")
y_test = pd.read_csv("y_test.csv").squeeze("columns")
```
#### Feature Scaling

Some regression models, such as **Ridge Regression** and **Lasso Regression**, require feature scaling. If you use one of these models, standardize the predictor variables **using only the training data**, then apply the same transformation to the test data.

```python
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
```

**Notes**

- Ordinary Linear Regression, Decision Trees, Random Forests, and HistGradientBoosting do **not** require feature scaling.
- If you create additional features later in this milestone and are using a scaled model, repeat the scaling step so the new features are transformed consistently.
- Throughout this milestone, use the same training/test split so that all models are evaluated on identical data.

In [8]:
# Add as many cells as you need
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final)
X_test_scaled  = scaler.transform(X_test_final)

## Problem 1: Model Selection and Baselines [6 pts]

### 1.A Coding

Select **three** regression models from the following list and evaluate each one using the cleaned training dataset.

Use the default hyperparameters provided by scikit-learn (except where scaling is required).

Available models:

* Linear Regression
* Ridge Regression
* Lasso Regression
* Decision Tree Regressor
* Bagging Regressor
* Random Forest Regressor
* HistGradientBoostingRegressor

For each of the three models you choose:

* Train using the **training dataset only**.
* Use **Repeated 5-Fold Cross-Validation** (5 repeats).
* Report validation performance:

  * Mean CV MAE
  * Standard Deviation of CV MAE

In [9]:
# Add as many code cells as needed.
#Model 1 Ridge

# Default Ridge model (alpha=1.0)
ridge = Ridge()

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring (sklearn returns negative MAE)
cv_mae_scores = cross_val_score(
    ridge,
    X_train_scaled,
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Mean CV MAE:", np.mean(cv_mae_scores))
print("Std Dev CV MAE:", np.std(cv_mae_scores))


Mean CV MAE: 241749.64936281185
Std Dev CV MAE: 3295.2413229666126


In [ ]:
#Model 2: Randomforest

# Default Random Forest (no scaling needed)
rf = RandomForestRegressor(random_state=42)

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring (sklearn returns negative MAE)
cv_mae_scores = cross_val_score(
    rf,
    X_train_final,   #use unscaled features
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Mean CV MAE:", np.mean(cv_mae_scores))
print("Std Dev CV MAE:", np.std(cv_mae_scores))


Mean CV MAE: 190868.85340780742
Std Dev CV MAE: 2469.257637435788


In [ ]:
#HistGradientBoostingRegressor

# Default HGB model (no scaling needed)
hgb = HistGradientBoostingRegressor(random_state=42)

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring (sklearn returns negative MAE)
cv_mae_scores = cross_val_score(
    hgb,
    X_train_final,   #unscaled features
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Mean CV MAE:", np.mean(cv_mae_scores))
print("Std Dev CV MAE:", np.std(cv_mae_scores))


Mean CV MAE: 192789.69515080936
Std Dev CV MAE: 2968.1996742366264


### 1.B Discussion

Answer the following questions.

#### 1.B.1

Which of your three models achieved the **lowest validation MAE score **?

> Randomforest achieved the lowest validation MAE score (Mean CV MAE: 190868.85340780742)

#### 1.B.2

Which model produced the **smallest standard deviation** across the repeated cross-validation runs? What does this suggest about its stability?

>RandomForestRegressor produced the smallest standard deviation of the cross-validation MAE (2,469.26) among the evaluated models. This indicates that its performance was the most consistent across the repeated cross-validation folds, suggesting that the model is stable and generalizes well to unseen data. Lower variability in MAE implies greater reliability and less sensitivity to different training and validation splits.

#### 1.B.3

Did any model appear to overfit or underfit? Explain your reasoning using the training and cross-validation results.

> None of the three models show evidence of severe overfitting. Overfitting would typically be indicated by very low training error, much higher cross-validation (CV) error, and high variability across folds. Although only CV results are available, the relatively low standard deviations across the 25 folds (approximately 2.4k-3.3k MAE) suggest that all models perform consistently on unseen data, with no signs of memorizing the training folds. However, Ridge Regression appears to underfit the data because it produced the highest mean CV MAE (241,750). As a linear model, Ridge cannot capture the complex nonlinear relationships and feature interactions common in housing price data, such as the combined effects of location, property size, quality, and neighborhood characteristics. Its slightly higher standard deviation across folds also indicates greater sensitivity to changes in the training data, which is consistent with a model that is too simple for the underlying problem.

#### 1.B.4

Compare the overall strengths and weaknesses of the three models. Did any model consistently perform better, or were there important tradeoffs between accuracy and stability?

> Across the three models evaluated, Random Forest Regressor delivered the best overall performance, achieving the lowest mean cross-validation MAE (190,869) and the lowest standard deviation (2,469), indicating both the highest predictive accuracy and the most consistent performance across the 25 cross-validation runs. HistGradientBoostingRegressor was a close second, with a slightly higher MAE (192,790) and greater variability (2,968), making it a strong but marginally less stable alternative. In contrast, Ridge Regression performed substantially worse, producing the highest MAE (241,750) and the greatest variability (3,295), which suggests that the linear model underfit the highly nonlinear relationships present in the Zillow dataset. Overall, none of the models showed evidence of overfitting, as cross-validation results remained stable across folds, but Ridge Regression clearly underfit the data. Therefore, Random Forest is the most accurate, stable, and reliable model for predicting Zillow home values.

## Part 2: Evaluate Your Feature Engineering Hypotheses [6 pts]

### 2.A Coding

In **Milestone 1**, you proposed several preprocessing and feature engineering ideas that you believed might improve predictive performance.

Select **at least three** of those ideas and evaluate them.

These may include, for example:

* Creating new features
* Transforming existing features
* Removing features
* Combining features
* Other preprocessing ideas that you proposed in Milestone 1

For each idea:

* Apply the preprocessing or feature engineering to the **training dataset only**.
* Retrain the same three baseline models from **Problem 1** using repeated 5-fold cross-validation (5 repeats).
* Compare the validation performance (mean CV MAE) and stability (standard deviation of CV MAE) with your original baseline results


> One of the most important things you can learn is that **not every clever idea results in an improvement**--they have to be evaluated by careful experiment.  And negative results are valuable if they are carefully evaluated and discussed!

In [ ]:
X_train_final.info()

In [10]:
# Add as many code cells as needed.
#Feature engineering part 1: Interaction Features
X_train_fe1 = X_train_final.copy()

X_train_fe1["sqft_quality"] = (
    X_train_fe1["calculatedfinishedsquarefeet"] *
    X_train_fe1["buildingqualitytypeid"]
)

X_train_fe1["bed_bath_ratio"] = (
    X_train_fe1["bedroomcnt"] / (X_train_fe1["bathroomcnt"] + 1)
)

Models using Feature engineering part 1: Interaction Features

In [ ]:
#Model 1 Ridge

#Need to add feature in test for scaling
#X_test_fe = X_test_final.copy()

#X_test_fe["sqft_quality"] = (
    #X_test_fe["calculatedfinishedsquarefeet"] *
    #X_test_fe["buildingqualitytypeid"]
#)

#X_test_fe["bed_bath_ratio"] = (
    #X_test_fe["bedroomcnt"] / (X_test_fe["bathroomcnt"] + 1)
#)

#X_test_scaled = scaler.transform(X_test_fe)

scaler = StandardScaler()
X_train_scaled1 = scaler.fit_transform(X_train_fe1)
#X_test_scaled = scaler.transform(X_test_fe)

# Default Ridge model (alpha=1.0)
ridge = Ridge()

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring (sklearn returns negative MAE)
cv_mae_scores = cross_val_score(
    ridge,
    X_train_scaled1,
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Mean CV MAE:", np.mean(cv_mae_scores))
print("Std Dev CV MAE:", np.std(cv_mae_scores))

Mean CV MAE: 236793.13480814724
Std Dev CV MAE: 3310.4232091971808


In [ ]:
#Model 2: Randomforest

# Default Random Forest (no scaling needed)
rf = RandomForestRegressor(random_state=42)

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring (sklearn returns negative MAE)
cv_mae_scores = cross_val_score(
    rf,
    X_train_fe1,   #use unscaled features
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Mean CV MAE:", np.mean(cv_mae_scores))
print("Std Dev CV MAE:", np.std(cv_mae_scores))


Mean CV MAE: 190575.49767716188
Std Dev CV MAE: 2528.8542388926808


In [ ]:
#HistGradientBoostingRegressor

# Default HGB model (no scaling needed)
hgb = HistGradientBoostingRegressor(random_state=42)

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring (sklearn returns negative MAE)
cv_mae_scores = cross_val_score(
    hgb,
    X_train_fe1,   #unscaled features
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Mean CV MAE:", np.mean(cv_mae_scores))
print("Std Dev CV MAE:", np.std(cv_mae_scores))


Mean CV MAE: 193169.36295022588
Std Dev CV MAE: 3266.3764897749734


In [10]:
#Feature engineering part 2: Log-transform skewed numeric features

X_train_fe2 = X_train_final.copy()

skewed_cols = [
    "lotsizesquarefeet",
    "calculatedfinishedsquarefeet"
]

for col in skewed_cols:
    X_train_fe2[col + "_log"] = np.log1p(X_train_fe2[col])

Models using Feature engineering part 2: Log-transform

In [ ]:
#Model 1 Ridge

#Need to add feature in test for scaling
#X_test_fe = X_test_final.copy()

#X_test_fe["sqft_quality"] = (
    #X_test_fe["calculatedfinishedsquarefeet"] *
    #X_test_fe["buildingqualitytypeid"]
#)

#X_test_fe["bed_bath_ratio"] = (
    #X_test_fe["bedroomcnt"] / (X_test_fe["bathroomcnt"] + 1)
#)

#X_test_scaled = scaler.transform(X_test_fe)

scaler = StandardScaler()
X_train_scaled2 = scaler.fit_transform(X_train_fe2)
#X_test_scaled = scaler.transform(X_test_fe)

# Default Ridge model (alpha=1.0)
ridge = Ridge()

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring (sklearn returns negative MAE)
cv_mae_scores = cross_val_score(
    ridge,
    X_train_scaled2,
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Mean CV MAE:", np.mean(cv_mae_scores))
print("Std Dev CV MAE:", np.std(cv_mae_scores))

Mean CV MAE: 234770.92762926553
Std Dev CV MAE: 3473.8111020922966


In [ ]:
#Model 2: Randomforest

# Default Random Forest (no scaling needed)
rf = RandomForestRegressor(random_state=42)

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring (sklearn returns negative MAE)
cv_mae_scores = cross_val_score(
    rf,
    X_train_fe2,   #use unscaled features
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Mean CV MAE:", np.mean(cv_mae_scores))
print("Std Dev CV MAE:", np.std(cv_mae_scores))


Mean CV MAE: 190888.7565498651
Std Dev CV MAE: 2483.437710426823


In [ ]:
#HistGradientBoostingRegressor

# Default HGB model (no scaling needed)
hgb = HistGradientBoostingRegressor(random_state=42)

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring (sklearn returns negative MAE)
cv_mae_scores = cross_val_score(
    hgb,
    X_train_fe2,   #unscaled features
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Mean CV MAE:", np.mean(cv_mae_scores))
print("Std Dev CV MAE:", np.std(cv_mae_scores))


Mean CV MAE: 192789.69515080936
Std Dev CV MAE: 2968.1996742366264


In [ ]:
#Feature engineering part 3: Remove noisy or low value features
X_train_fe3 = X_train_final.drop(columns=[
    "propertyzoningdesc",
    "taxdelinquencyflag",
    "fireplaceflag"
])

Models using Feature engineering part 3: Remove features

In [ ]:
#Model 1 Ridge

#Need to add feature in test for scaling
#X_test_fe = X_test_final.copy()

#X_test_fe["sqft_quality"] = (
    #X_test_fe["calculatedfinishedsquarefeet"] *
    #X_test_fe["buildingqualitytypeid"]
#)

#X_test_fe["bed_bath_ratio"] = (
    #X_test_fe["bedroomcnt"] / (X_test_fe["bathroomcnt"] + 1)
#)

#X_test_scaled = scaler.transform(X_test_fe)

scaler = StandardScaler()
X_train_scaled3 = scaler.fit_transform(X_train_fe3)
#X_test_scaled = scaler.transform(X_test_fe)

# Default Ridge model (alpha=1.0)
ridge = Ridge()

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring (sklearn returns negative MAE)
cv_mae_scores = cross_val_score(
    ridge,
    X_train_scaled3,
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Mean CV MAE:", np.mean(cv_mae_scores))
print("Std Dev CV MAE:", np.std(cv_mae_scores))

Mean CV MAE: 241731.43430856857
Std Dev CV MAE: 3303.724187046125


In [ ]:
#Model 2: Randomforest

# Default Random Forest (no scaling needed)
rf = RandomForestRegressor(random_state=42)

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring (sklearn returns negative MAE)
cv_mae_scores = cross_val_score(
    rf,
    X_train_fe3,   #use unscaled features
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Mean CV MAE:", np.mean(cv_mae_scores))
print("Std Dev CV MAE:", np.std(cv_mae_scores))


Mean CV MAE: 191052.3154964035
Std Dev CV MAE: 2477.1190795854804


In [ ]:
#HistGradientBoostingRegressor

# Default HGB model (no scaling needed)
hgb = HistGradientBoostingRegressor(random_state=42)

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring (sklearn returns negative MAE)
cv_mae_scores = cross_val_score(
    hgb,
    X_train_fe3,   #unscaled features
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Mean CV MAE:", np.mean(cv_mae_scores))
print("Std Dev CV MAE:", np.std(cv_mae_scores))


Mean CV MAE: 192886.098487441
Std Dev CV MAE: 2977.861544660217


### 2.B Discussion

Answer the following questions.

#### 2.B.1

Which of your feature engineering ideas produced the largest improvement in validation performance?

> Across all three featur engineering ideas are tested, log-transforming skewed numeric features produced the largest overall improvement in validation performance, especially for **Ridge** and **HistGradientBoostingRegressor**, while **RandomForest** remained consistently strong and stable across all three.

#### 2.B.2

Were any of your ideas unsuccessful or did they reduce model performance? Briefly explain.

> Removing selected features did not improve model performance and slightly reduced accuracy for the tree-based models. The Random Forest model's MAE increased from 190,868 to 191,052 (+184), while the Histogram Gradient Boosting model's MAE increased from 192,790 to 192,886 (+96). The Ridge regression model showed virtually no change, with its MAE decreasing marginally from 241,749 to 241,731. These results suggest that the removed features, particularly location-related attributes, still contained useful predictive information. Tree-based models are especially effective at leveraging weak or noisy features, so eliminating them reduced their ability to capture important patterns. Overall, feature removal did not provide any performance benefit and slightly worsened the results for the best-performing models.

#### 2.B.3

Did some models benefit more from feature engineering than others? If so, why do you think this occurred?

> The **Ridge Regression **model benefited the most from feature engineering, with the log transformation reducing the MAE by nearly 7,000. This is because Ridge is a linear model that assumes a linear relationship between the predictors and the target. Log-transforming skewed variables makes relationships more linear and reduces the influence of extreme values, allowing the model to fit the data more effectively.

>The **RandomForest** model showed only a very small improvement (about 293 MAE) from adding interaction features. Random Forests naturally capture nonlinear relationships and interactions between variables through their tree structure, so manually creating interaction features provides only a limited additional benefit.

>The **HistGradientBoostingRegressor** did not show any meaningful improvement over the baseline. Like Random Forests, gradient boosting trees can automatically model complex nonlinear relationships and feature interactions. As a result, the engineered features offered little additional information beyond what the model could already learn from the original features.

>Overall, feature engineering had the greatest impact on the linear model (Ridge), while the tree-based models benefited very little because they already handle nonlinearities and interactions internally.

#### 2.B.4

Which preprocessing or feature engineering changes will you keep for the remainder of the milestone? Briefly justify your decision.

> For the remainder of the milestone, I will keep the log transformation of the skewed features. It consistently improved the performance of the Ridge Regression model, reducing the cross-validation MAE by about 7,000, improved the stability of RandomForest and HistGradientBoostingRegressor and it did not negatively affect the tree-based models. Log transformation also reduces skewness, minimizes the influence of extreme values, and makes the data more suitable for linear models.

## Part 3: Refine the Feature Set [6 pts]

### 3.A Coding

Using your dataset after completing **Part 2** (including any preprocessing and feature engineering changes you decided to keep):

Investigate whether **feature selection** can further improve model performance.

You may use one or more of the following methods:

* Forward Selection (for linear regression models)
* Backward Selection (for linear regression models)
* Feature importance from tree-based models (for decision trees, Random Forests, Bagging, and HistGradientBoosting)
* Another reasonable feature selection method

For each of your three models:

* Select a subset of features using an appropriate feature selection method.
* Retrain the model using only the selected features.
* Evaluate the model using the same repeated cross-validation procedure as before.
* Report the validation performance (the mean and standard deviation of the CV MAE).

> Not every model will necessarily benefit from feature selection. Choose methods that are appropriate for the models you selected. Negative results are valuable if they are carefully evaluated and discussed!

In [12]:
X_train_fe2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 62062 entries, 42412 to 15805
Data columns (total 51 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   bathroomcnt                       62062 non-null  float64
 1   bedroomcnt                        62062 non-null  float64
 2   calculatedbathnbr                 62062 non-null  float64
 3   finishedfloor1squarefeet          62062 non-null  float64
 4   calculatedfinishedsquarefeet      62062 non-null  float64
 5   finishedsquarefeet12              62062 non-null  float64
 6   finishedsquarefeet50              62062 non-null  float64
 7   fireplacecnt                      62062 non-null  float64
 8   fullbathcnt                       62062 non-null  float64
 9   garagecarcnt                      62062 non-null  float64
 10  garagetotalsqft                   62062 non-null  float64
 11  latitude                          62062 non-null  float64
 12  longi

In [12]:
# Add as many code cells as needed.
#Model 1: Ridge Regression with forward selection

# Start with log-transformed dataset
X = X_train_fe2.copy()
y = y_train.copy()

# Scale features for Ridge
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Forward Selection
selected_features = []
remaining_features = list(X.columns)
best_mae = np.inf

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

while remaining_features:
    mae_candidates = []

    for feature in remaining_features:
        trial_features = selected_features + [feature]
        X_trial = X_scaled[:, [X.columns.get_loc(f) for f in trial_features]]

        ridge = Ridge()
        scores = -cross_val_score(
            ridge, X_trial, y, cv=cv, scoring='neg_mean_absolute_error'
        )
        mae_candidates.append((feature, scores.mean()))

    # Select best feature
    best_feature, best_feature_mae = min(mae_candidates, key=lambda x: x[1])

    if best_feature_mae < best_mae:
        selected_features.append(best_feature)
        remaining_features.remove(best_feature)
        best_mae = best_feature_mae
    else:
        break

print("Selected features:", selected_features)
print("Best Ridge MAE:", best_mae)


Selected features: ['calculatedfinishedsquarefeet', 'calculatedfinishedsquarefeet_log', 'latitude', 'regionidneighborhood', 'finishedsquarefeet12', 'bathroomcnt', 'propertycountylandusecode', 'heatingorsystemtypeid_24.0', 'lotsizesquarefeet_log', 'airconditioningtypeid_13.0', 'airconditioningtypeid_11.0', 'heatingorsystemtypeid_18.0', 'airconditioningtypeid_Unknown', 'buildingqualitytypeid', 'garagecarcnt', 'garagetotalsqft', 'propertylandusetypeid', 'heatingorsystemtypeid_7.0', 'bedroomcnt', 'finishedfloor1squarefeet', 'numberofstories', 'airconditioningtypeid_1.0']
Best Ridge MAE: 232093.05612228555


In [13]:
#Final evaluation using selected features
X_fs_scaled = X_scaled[:, [X.columns.get_loc(f) for f in selected_features]]

ridge_final = Ridge()
cv_mae_scores = -cross_val_score(
    ridge_final,
    X_fs_scaled,
    y,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

print("Ridge (Forward Selection) Mean CV MAE:", np.mean(cv_mae_scores))
print("Ridge (Forward Selection) Std Dev CV MAE:", np.std(cv_mae_scores))

Ridge (Forward Selection) Mean CV MAE: 232093.05612228555
Ridge (Forward Selection) Std Dev CV MAE: 2916.9930813041005


In [14]:
#Model 2: RandomForest with Tree based feature importance
# Use log-transformed dataset (no scaling needed)
X = X_train_fe2.copy()
y = y_train.copy()

# Train RF to get feature importance
rf_full = RandomForestRegressor(random_state=42)
rf_full.fit(X, y)

# Rank features
importances = pd.Series(rf_full.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=False).head(25).index.tolist()

print("Selected RF features:", top_features)

# Evaluate RF using selected features
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)
rf = RandomForestRegressor(random_state=42)

scores = -cross_val_score(
    rf,
    X[top_features],
    y,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

print("RF Mean CV MAE:", scores.mean())
print("RF Std Dev CV MAE:", scores.std())


Selected RF features: ['finishedsquarefeet12', 'latitude', 'calculatedfinishedsquarefeet', 'calculatedfinishedsquarefeet_log', 'longitude', 'yearbuilt', 'buildingqualitytypeid', 'lotsizesquarefeet_log', 'lotsizesquarefeet', 'propertyzoningdesc', 'regionidzip', 'bedroomcnt', 'rawcensustractandblock', 'regionidneighborhood', 'censustractandblock', 'bathroomcnt', 'regionidcity', 'propertycountylandusecode', 'calculatedbathnbr', 'garagetotalsqft', 'heatingorsystemtypeid_20.0', 'fullbathcnt', 'propertylandusetypeid', 'airconditioningtypeid_Unknown', 'roomcnt']
RF Mean CV MAE: 191037.5432551268
RF Std Dev CV MAE: 2491.691205789419


In [11]:
from sklearn.inspection import permutation_importance

#Model 2: HistGradientBoosting with permutation_importance
# Use log-transformed dataset
X = X_train_fe2.copy()
y = y_train.copy()

# Train full HGB model
hgb_full = HistGradientBoostingRegressor(random_state=42)
hgb_full.fit(X, y)

# Compute permutation importance
perm = permutation_importance(
    hgb_full,
    X,
    y,
    n_repeats=5,
    random_state=42
)

importances = pd.Series(perm.importances_mean, index=X.columns)

# Select top 20 features
top_features = importances.sort_values(ascending=False).head(20).index.tolist()
print("Selected HGB features:", top_features)

# Evaluate HGB using selected features
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)
hgb = HistGradientBoostingRegressor(random_state=42)

scores = -cross_val_score(
    hgb,
    X[top_features],
    y,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

print("HGB Mean CV MAE:", scores.mean())
print("HGB Std Dev CV MAE:", scores.std())


Selected HGB features: ['latitude', 'longitude', 'finishedsquarefeet12', 'calculatedfinishedsquarefeet', 'yearbuilt', 'lotsizesquarefeet', 'bathroomcnt', 'buildingqualitytypeid', 'rawcensustractandblock', 'bedroomcnt', 'propertycountylandusecode', 'regionidneighborhood', 'regionidcity', 'regionidzip', 'propertyzoningdesc', 'propertylandusetypeid', 'censustractandblock', 'fullbathcnt', 'airconditioningtypeid_1.0', 'garagetotalsqft']
HGB Mean CV MAE: 192544.67037713586
HGB Std Dev CV MAE: 2942.1117140725078


### 3.B Discussion

#### 3.B.1

Did feature selection improve the validation performance of any of your models?

>Feature selection was most beneficial for Ridge Regression

> Feature selection had different effects depending on the model. It improved Ridge Regression, reducing the cross-validated MAE from approximately 234,771 to 232,093 while also lowering the standard deviation 3,474 to 2,917, indicating more stable performance. This is expected because Ridge Regression benefits from removing irrelevant or noisy features.

> For Random Forest, feature selection did not improve performance. The MAE increased slightly from approximately 190,889 to 191,038, with almost no change 2,483 to 2,492 in stability. This suggests that Random Forest already handles irrelevant features effectively and performs best with the full feature set.

> For HistGradientBoosting, feature selection produced only a minor improvement, reducing the MAE from approximately 192,790 to 192,545, with a slight improvement 2,968 to 2,942 in stability. The gain was minimal because the model was already performing well after the log transformation.


#### 3.B.2

Were there features that were consistently retained (or consistently removed) across multiple models?

> The following features were consistently identified as uninformative across all three models(Ridge, Random Forest, and HistGradientBoosting) and removed from the final feature set:

>finishedsquarefeet50
>fireplacecnt
>poolsizesum
>threequarterbathnbr
>unitcnt
>yardbuildingsqft17
>airconditioningtypeid_5.0
>airconditioningtypeid_9.0
>heatingorsystemtypeid_1.0
>heatingorsystemtypeid_10.0
>heatingorsystemtypeid_11.0
>heatingorsystemtypeid_13.0
>heatingorsystemtypeid_2.0
>heatingorsystemtypeid_6.0
>heatingorsystemtypeid_Unknown
>fireplaceflag
>taxdelinquencyflag

>Interpretation: Since these features were not selected by any of the three models, they appear to contribute little or no predictive value for estimating home prices. Their lack of importance likely reflects weak relationships with the target variable, high sparsity, or noisy information. Removing them simplifies the model, reduces dimensionality, and improves efficiency without significantly affecting predictive performance.

>The features that consistently appeared across Ridge, Random Forest, and HistGradientBoosting are the strongest predictors of home value. These include:

> Square footage: finishedsquarefeet12, calculatedfinishedsquarefeet
> Bathrooms: bathroomcnt
> Bedrooms: bedroomcnt
> Location: latitude, regionidneighborhood
> Land use: propertycountylandusecode, propertylandusetypeid
> Garage size: garagetotalsqft
>Building quality: buildingqualitytypeid

>These variables consistently demonstrated high predictive power because they directly capture the key factors that influence a property's market value. Larger homes with more living space, bedrooms, and bathrooms generally command higher prices, while neighborhood and geographic location reflect differences in demand, accessibility, school districts, and local amenities. Building quality indicates the condition and construction standard of the property, and garage size adds functional and resale value. Land use codes distinguish different property types, which have different pricing patterns. Since these features were selected by multiple modeling approaches—including both linear and tree-based models—they exhibit robust, model-independent predictive strength and should be retained in the final modeling pipeline.

#### 3.B.3

Were any of your engineered features selected as important? If so, what does this suggest about the hypotheses you developed in Milestone 1?

> Ridge Regression selected several engineered features, indicating that feature engineering improved the model's ability to capture linear relationships with property values. Both log-transformed variables—calculatedfinishedsquarefeet_log and lotsizesquarefeet_log—were identified as important, suggesting that applying logarithmic transformations reduced skewness and created more linear relationships with the target variable. Ridge also selected multiple one-hot encoded categorical features related to air conditioning and heating system types, demonstrating that converting categorical variables into numerical indicators provided additional predictive information. Overall, these results show that Ridge benefits from engineered features because they enhance the representation of the data and improve the model's ability to learn meaningful linear patterns.

>The results provide partial support for the Milestone 1 hypothesis, demonstrating that the effectiveness of feature engineering depends on the type of machine learning model. The hypothesis was strongly validated for Ridge Regression, where the log-transformed features were consistently selected, leading to improved performance and model stability. This confirms that reducing skewness and creating more linear relationships benefits linear models, which rely on these assumptions. However, the hypothesis was not supported for Random Forest or HistGradientBoosting, as neither model selected the log-transformed features and instead favored the original continuous variables. These tree-based models naturally handle skewed data, outliers, and non-linear relationships through threshold-based splits, making log transformations largely unnecessary. Overall, the findings indicate that log transformations are model-specific rather than universally beneficial, providing clear advantages for linear models like Ridge Regression but offering little or no benefit for tree-based ensemble methods.

#### 3.B.4

After feature selection, did simpler models perform as well as—or better than—the models using the full feature set? Briefly discuss any tradeoffs you observed between model complexity and predictive performance.

> Feature selection and preprocessing had different effects depending on the model. The Ridge Regression model benefited the most, with its cross-validation MAE improving from 241,750 to 232,093 and its standard deviation decreasing from 3,295 to 2,917. This indicates that removing less informative features and applying log transformations reduced noise, strengthened linear relationships, and improved the model's ability to generalize. Since Ridge is a linear model, it performs better with a smaller set of relevant predictors and less multicollinearity.

> The Random Forest model experienced a slight decline in performance, with MAE increasing from 190,869 to 191,038. This is expected because Random Forests naturally handle irrelevant features and learn complex nonlinear relationships and interactions. Removing features may have eliminated variables that individually appeared weak but collectively provided useful predictive information, slightly reducing the model's flexibility.

> The HistGradientBoostingRegressor showed only a very small improvement, with MAE decreasing from 192,790 to 192,545 and a slight reduction in standard deviation. Histogram-based gradient boosting already manages skewed data, nonlinear relationships, and feature interactions effectively, so additional feature selection had only a minimal effect on its performance.

> Overall, these results highlight that the effectiveness of feature selection depends on the model. Simpler linear models, such as Ridge Regression, can benefit significantly from reducing the feature set and improving feature distributions, while more complex tree-based models generally require less manual feature engineering because they can automatically identify useful splits and interactions. In this project, feature selection and preprocessing clearly improved the Ridge model but provided little to no benefit for the Random Forest and HistGradientBoostingRegressor models.

## Part 4: Tune Your Models [8 pts]

### 4.A Coding

Using the three models developed in **Part 3** (including your final preprocessing, feature engineering, and feature selection decisions):

Investigate whether **hyperparameter tuning** can further improve model performance.

For each of your three models:

* Select one or more important hyperparameters to tune.
* Use one or more appropriate tuning methods. Consider first using validation curves (`sweep_parameter`) to identify a promising region or performance plateau, followed by a focused search using methods such as:

    * GridSearchCV
    * RandomizedSearchCV
    * Another reasonable hyperparameter search method

* Choose hyperparameter values based on the validation results. If several nearby values produce similar validation performance (a performance plateau), prefer **values near the beginning of the plateau,** since they often produce simpler models with nearly identical predictive performance.
* Retrain the model using those hyperparameters.
* Evaluate the tuned model using repeated 5-fold cross-validation (5 repeats).
* Report the validation performance (**mean** and **standard deviation** of the CV MAE).


In [11]:

# Model 1: Ridge Regression with Forward Selection + Tuning
from sklearn.model_selection import (
    RepeatedKFold, cross_val_score, validation_curve, GridSearchCV
)

# 1. Use log-transformed + engineered dataset
X = X_train_fe2.copy()
y = y_train.copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Forward Selection
selected_features = []
remaining_features = list(X.columns)
best_mae = np.inf

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

while remaining_features:
    mae_candidates = []

    for feature in remaining_features:
        trial_features = selected_features + [feature]
        X_trial = X_scaled[:, [X.columns.get_loc(f) for f in trial_features]]

        ridge = Ridge()
        scores = -cross_val_score(
            ridge, X_trial, y, cv=cv, scoring='neg_mean_absolute_error'
        )
        mae_candidates.append((feature, scores.mean()))

    best_feature, best_feature_mae = min(mae_candidates, key=lambda x: x[1])

    if best_feature_mae < best_mae:
        selected_features.append(best_feature)
        remaining_features.remove(best_feature)
        best_mae = best_feature_mae
    else:
        break

print("Selected features:", selected_features)
print("Best Ridge MAE (before tuning):", best_mae)

# 3. SWEEP PARAMETER (Validation Curve)

X_fs_scaled = X_scaled[:, [X.columns.get_loc(f) for f in selected_features]]

alpha_range = np.logspace(-4, 4, 30)  # wide sweep

train_scores, val_scores = validation_curve(
    Ridge(),
    X_fs_scaled,
    y,
    param_name="alpha",
    param_range=alpha_range,
    cv=cv,
    scoring="neg_mean_absolute_error"
)

mean_val_mae = -val_scores.mean(axis=1)

best_alpha_index = np.argmin(mean_val_mae)
best_alpha_from_curve = alpha_range[best_alpha_index]

print("Best alpha from sweep_parameter (validation curve):", best_alpha_from_curve)


# 4. Focused GridSearchCV around plateau
alpha_grid = [
    best_alpha_from_curve / 3,
    best_alpha_from_curve / 2,
    best_alpha_from_curve,
    best_alpha_from_curve * 2,
    best_alpha_from_curve * 3
]

param_grid = {"alpha": alpha_grid}

ridge = Ridge()

grid_search = GridSearchCV(
    ridge,
    param_grid,
    cv=cv,
    scoring="neg_mean_absolute_error"
)

grid_search.fit(X_fs_scaled, y)

best_alpha_final = grid_search.best_params_["alpha"]
print("Final tuned alpha:", best_alpha_final)

# 5. Final evaluation using tuned hyperparameters
ridge_final = Ridge(alpha=best_alpha_final)

cv_mae_scores = -cross_val_score(
    ridge_final,
    X_fs_scaled,
    y,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

print("Ridge (Tuned + FS) Mean CV MAE:", np.mean(cv_mae_scores))
print("Ridge (Tuned + FS) Std Dev CV MAE:", np.std(cv_mae_scores))


Selected features: ['calculatedfinishedsquarefeet', 'calculatedfinishedsquarefeet_log', 'latitude', 'regionidneighborhood', 'finishedsquarefeet12', 'bathroomcnt', 'propertycountylandusecode', 'heatingorsystemtypeid_24.0', 'lotsizesquarefeet_log', 'airconditioningtypeid_13.0', 'airconditioningtypeid_11.0', 'heatingorsystemtypeid_18.0', 'airconditioningtypeid_Unknown', 'buildingqualitytypeid', 'garagecarcnt', 'garagetotalsqft', 'propertylandusetypeid', 'heatingorsystemtypeid_7.0', 'bedroomcnt', 'finishedfloor1squarefeet', 'numberofstories', 'airconditioningtypeid_1.0']
Best Ridge MAE (before tuning): 232093.05612228555
Best alpha from sweep_parameter (validation curve): 2807.2162039411755
Final tuned alpha: 2807.2162039411755
Ridge (Tuned + FS) Mean CV MAE: 230831.52328677822
Ridge (Tuned + FS) Std Dev CV MAE: 2739.8720072103774


In [ ]:
# ============================================================
# FAST RandomForest: Feature Selection + sweep_parameter + tiny GridSearchCV
# ============================================================

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import (
    RepeatedKFold, cross_val_score, validation_curve, GridSearchCV
)

# ------------------------------------------------------------
# 1. Feature Selection using Tree-Based Importance
# ------------------------------------------------------------
X = X_train_fe2.copy()
y = y_train.copy()

rf_full = RandomForestRegressor(
    n_estimators=150,   # fixed to keep runtime low
    random_state=42
)
rf_full.fit(X, y)

importances = pd.Series(rf_full.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=False).head(25).index.tolist()

print("Selected RF features:", top_features)

X_fs = X[top_features]

# ------------------------------------------------------------
# 2. SWEEP PARAMETER (Validation Curve)
# ------------------------------------------------------------
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

depth_range = [5, 10, 15, 20, 25]

train_scores, val_scores = validation_curve(
    RandomForestRegressor(n_estimators=150, random_state=42),
    X_fs,
    y,
    param_name="max_depth",
    param_range=depth_range,
    cv=cv,
    scoring="neg_mean_absolute_error"
)

mean_val_mae = -val_scores.mean(axis=1)
best_depth = depth_range[np.argmin(mean_val_mae)]

print("Best max_depth from sweep_parameter:", best_depth)

# ------------------------------------------------------------
# 3. Tiny GridSearchCV (FAST)
# ------------------------------------------------------------
param_grid = {
    "max_depth": [best_depth - 5, best_depth, best_depth + 5],
    "max_features": ["sqrt", None]
}

rf = RandomForestRegressor(
    n_estimators=150,
    random_state=42
)

grid_search = GridSearchCV(
    rf,
    param_grid,
    cv=cv,
    scoring="neg_mean_absolute_error"
)

grid_search.fit(X_fs, y)

best_params = grid_search.best_params_
print("Tuned RF hyperparameters:", best_params)

# ------------------------------------------------------------
# 4. Final Evaluation using Tuned Hyperparameters
# ------------------------------------------------------------
rf_final = RandomForestRegressor(
    n_estimators=150,
    **best_params,
    random_state=42
)

cv_mae_scores = -cross_val_score(
    rf_final,
    X_fs,
    y,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

print("RF (Tuned + FS) Mean CV MAE:", np.mean(cv_mae_scores))
print("RF (Tuned + FS) Std Dev CV MAE:", np.std(cv_mae_scores))


Selected RF features: ['finishedsquarefeet12', 'latitude', 'calculatedfinishedsquarefeet', 'calculatedfinishedsquarefeet_log', 'longitude', 'yearbuilt', 'buildingqualitytypeid', 'lotsizesquarefeet_log', 'lotsizesquarefeet', 'propertyzoningdesc', 'regionidzip', 'bedroomcnt', 'regionidneighborhood', 'rawcensustractandblock', 'censustractandblock', 'regionidcity', 'propertycountylandusecode', 'bathroomcnt', 'calculatedbathnbr', 'heatingorsystemtypeid_20.0', 'garagetotalsqft', 'fullbathcnt', 'airconditioningtypeid_Unknown', 'propertylandusetypeid', 'roomcnt']
Best max_depth from sweep_parameter: 20


In [ ]:
# Model 3: HistGradientBoostingRegressor + Feature Selection + Tuning

# 1. Use engineered dataset
X = X_train_fe2.copy()
y = y_train.copy()

# 2. Feature Selection using Permutation Importance
hgb_full = HistGradientBoostingRegressor(random_state=42)
hgb_full.fit(X, y)

perm = permutation_importance(
    hgb_full,
    X,
    y,
    n_repeats=5,
    random_state=42
)

importances = pd.Series(perm.importances_mean, index=X.columns)

top_features = importances.sort_values(ascending=False).head(20).index.tolist()
print("Selected HGB features:", top_features)

X_fs = X[top_features]

# 3. SWEEP PARAMETER (Validation Curve)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# Sweep learning_rate
lr_range = np.logspace(-3, 0, 15)

train_scores, val_scores = validation_curve(
    HistGradientBoostingRegressor(random_state=42),
    X_fs,
    y,
    param_name="learning_rate",
    param_range=lr_range,
    cv=cv,
    scoring="neg_mean_absolute_error"
)

mean_val_mae = -val_scores.mean(axis=1)

best_index = np.argmin(mean_val_mae)
best_lr_from_curve = lr_range[best_index]

print("Best learning_rate from sweep_parameter:", best_lr_from_curve)

# 4. Focused Hyperparameter Tuning (RandomizedSearchCV)
param_dist = {
    "learning_rate": [
        best_lr_from_curve / 3,
        best_lr_from_curve / 2,
        best_lr_from_curve,
        best_lr_from_curve * 2
    ],
    "max_depth": [3, 5, 7, None],
    "max_leaf_nodes": [15, 31, 63],
    "min_samples_leaf": [1, 5, 10]
}

hgb = HistGradientBoostingRegressor(random_state=42)

random_search = RandomizedSearchCV(
    hgb,
    param_distributions=param_dist,
    n_iter=20,
    cv=cv,
    scoring="neg_mean_absolute_error",
    random_state=42
)

random_search.fit(X_fs, y)

best_params = random_search.best_params_
print("Tuned HGB hyperparameters:", best_params)

# 5. Final Evaluation using Tuned Hyperparameters
hgb_final = HistGradientBoostingRegressor(**best_params, random_state=42)

cv_mae_scores = -cross_val_score(
    hgb_final,
    X_fs,
    y,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

print("HGB (Tuned + FS) Mean CV MAE:", np.mean(cv_mae_scores))
print("HGB (Tuned + FS) Std Dev CV MAE:", np.std(cv_mae_scores))


### 4.B Discussion

Answer the following questions.

#### 4.B.1

Which hyperparameters had the greatest impact on model performance? Briefly explain.

> Replace this text with your answer.

#### 4.B.2

Did hyperparameter tuning substantially improve the performance of all three models, or only some of them?

> Replace this text with your answer.

#### 4.B.3

Which tuning method(s) did you use for each model? Briefly explain why you chose those methods.

> Replace this text with your answer.

#### 4.B.4

After tuning, how did the relative performance of your three models change? Did tuning affect which model appeared to perform best?

> Replace this text with your answer.

## Part 5: Final Model and Workflow Assessment [14 pts]

### 5.A Coding

Using the work completed in **Parts 1–4**:

Select your **best-performing model** and prepare your final modeling pipeline.

Your pipeline should include all preprocessing, feature engineering, feature selection, and hyperparameter tuning decisions that you chose to retain.

Evaluate your final model by:

* Training on the complete training dataset.
* Reporting the **mean** and **standard deviation** of the repeated cross-validation MAE.
* Evaluating the model on the held-out test set.
* Reporting the final test MAE.

In [ ]:
# Add as many code cells as needed.

### 5.B Discussion

Answer the following questions.

#### 5.B.1

Compare the performance of your final model with its original baseline from **Part 1**. Which changes contributed the most to the improvement?

> Replace this text with your answer.

#### 5.B.2

Looking back at the hypotheses you proposed in **Milestone 1**, which were supported by your experimental results? Were any hypotheses disproved?

> Replace this text with your answer.

#### 5.B.3

Why did you select this model as your final model? Discuss both its predictive performance and any other considerations (such as stability, simplicity, or interpretability).

> Replace this text with your answer.

#### 5.B.4

What did you learn about your dataset and the machine learning process through this end-to-end modeling workflow? If you had additional time, what would you investigate next?

> Replace this text with your answer.